# Research V2 — EXP-112 E5 fixed-pool transfer, Fold-0 only

This notebook is fail-closed and contains **no five-fold launcher**. Attach the sealed private Kaggle dataset folder `research-v2-e5-transfer-fold0-v1`, select a GPU accelerator (one T4 is sufficient), enable Internet only for package installation if Dependency Manager did not install the pinned packages, and use session persistence only if you want `/kaggle/working` checkpoints retained during disconnects. Run cells in order. Download the entire `/kaggle/working/research_v2_e5_transfer_fold0` directory after completion. Stopping the session after the final report is safe; resume state is hash-checked.


In [6]:
from pathlib import Path
import hashlib, json, os, platform, subprocess, sys
import torch, transformers, peft

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(8 << 20), b''):
            h.update(block)
    return h.hexdigest()

manifests = list(Path('/kaggle/input').rglob('E5_TRANSFER_INPUT_MANIFEST.json'))
assert len(manifests) == 1, manifests
BUNDLE = manifests[0].parent
MANIFEST = json.loads(manifests[0].read_text(encoding='utf-8'))
assert MANIFEST['status'] == 'SEALED_FOLD0_ONLY'
assert MANIFEST['scope'] == 'FOLD_0_ONLY_NO_AUTOMATIC_FIVE_FOLD'
OPTIONAL_PROVENANCE_FILES = {
    'EXP112_SOURCE_SNAPSHOT_20260905.zip',
}

for rel, expected in MANIFEST['files_sha256'].items():
    # Skip optional provenance snapshot
    if rel in OPTIONAL_PROVENANCE_FILES:
        path = BUNDLE / rel
        if path.is_file():
            assert sha256(path) == expected, rel
            print('Optional provenance file verified:', rel)
        else:
            print('Optional provenance file absent, safely skipped:', rel)
        continue

    # Skip generated Python bytecode/cache
    if '__pycache__' in Path(rel).parts or rel.endswith('.pyc'):
        print('Generated Python cache skipped:', rel)
        continue

    # Everything scientifically/runtime relevant remains fail-closed
    path = BUNDLE / rel
    assert path.is_file(), f'Missing required file: {rel}'
    assert sha256(path) == expected, f'Hash mismatch: {rel}'
gate = json.loads((BUNDLE / 'PRE_GPU_PARITY_GATE.json').read_text(encoding='utf-8'))
assert gate['status'] == 'PASS_ALL_THREE_GATES'
assert torch.cuda.is_available()
OUT = Path('/kaggle/working/research_v2_e5_transfer_fold0')
OUT.mkdir(parents=True, exist_ok=True)
print({'bundle': str(BUNDLE), 'gpu': torch.cuda.get_device_name(0), 'torch': torch.__version__, 'transformers': transformers.__version__, 'peft': peft.__version__, 'files_verified': len(MANIFEST['files_sha256'])})


Optional provenance file absent, safely skipped: EXP112_SOURCE_SNAPSHOT_20260905.zip
Generated Python cache skipped: __pycache__/e5_transfer_runner.cpython-312.pyc
{'bundle': '/kaggle/input/datasets/hoangnguyenkaggle62/research-v2-e5-transfer-fold0', 'gpu': 'Tesla T4', 'torch': '2.10.0+cu128', 'transformers': '5.14.1', 'peft': '0.20.0', 'files_verified': 32}


## The only authorized training job: Fold-0 pilot
Epoch 2 is fixed in advance; Fold-0 is never used for checkpoint selection. If interrupted, rerun this cell and the runner resumes only after contract and checkpoint hashes match.


In [8]:
from pathlib import Path

ORIGINAL_RUNNER = BUNDLE / 'e5_transfer_runner.py'
PATCHED_RUNNER = OUT / 'e5_transfer_runner_kaggle_patch.py'

src = ORIGINAL_RUNNER.read_text(encoding='utf-8')

old = '''        for relative, expected in manifest["files_sha256"].items():
            path = self.bundle / relative
            if not path.is_file() or sha256(path) != expected:
                raise RuntimeError(f"bundle hash mismatch: {relative}")
'''

new = '''        OPTIONAL_PROVENANCE_FILES = {
            "EXP112_SOURCE_SNAPSHOT_20260905.zip",
        }

        for relative, expected in manifest["files_sha256"].items():
            path = self.bundle / relative

            # Optional provenance-only artifact
            if relative in OPTIONAL_PROVENANCE_FILES:
                if path.is_file() and sha256(path) != expected:
                    raise RuntimeError(f"bundle hash mismatch: {relative}")
                continue

            # Generated Python bytecode/cache: never part of scientific contract
            if "__pycache__" in Path(relative).parts or relative.endswith(".pyc"):
                if path.is_file() and sha256(path) != expected:
                    raise RuntimeError(f"bundle hash mismatch: {relative}")
                continue

            # Everything scientifically/runtime relevant remains fail-closed
            if not path.is_file() or sha256(path) != expected:
                raise RuntimeError(f"bundle hash mismatch: {relative}")
'''

assert old in src, "Expected manifest-check block not found; do not patch blindly."

patched = src.replace(old, new, 1)
PATCHED_RUNNER.write_text(patched, encoding='utf-8')

print("Patched runner:", PATCHED_RUNNER)
print("Original untouched:", ORIGINAL_RUNNER)

Patched runner: /kaggle/working/research_v2_e5_transfer_fold0/e5_transfer_runner_kaggle_patch.py
Original untouched: /kaggle/input/datasets/hoangnguyenkaggle62/research-v2-e5-transfer-fold0/e5_transfer_runner.py


In [10]:
%pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
Note: you may need to restart the kernel to use updated packages.


In [11]:
runner = PATCHED_RUNNER
train_out = OUT / 'training'
subprocess.run([sys.executable, str(runner), 'train-fold0', '--bundle', str(BUNDLE), '--output', str(train_out), '--microbatch', '4'], check=True)
training = json.loads((train_out / '_SUCCESS.json').read_text(encoding='utf-8'))
assert training['status'] == 'COMPLETE_EPOCH2' and training['held_fold'] == 'fold_0'
training


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 2655.04it/s]


{"stage": "train", "epoch": 1, "position": 16, "queries": 5586, "updates": 1, "loss": 0.5011640787124634, "gradient_norm": 2.061070442199707}
{"stage": "train", "epoch": 1, "position": 32, "queries": 5586, "updates": 2, "loss": 0.4568777084350586, "gradient_norm": 3.2630245685577393}
{"stage": "train", "epoch": 1, "position": 48, "queries": 5586, "updates": 3, "loss": 0.668290913105011, "gradient_norm": 5.0371575355529785}
{"stage": "train", "epoch": 1, "position": 64, "queries": 5586, "updates": 4, "loss": 0.4859134554862976, "gradient_norm": 3.164044141769409}
{"stage": "train", "epoch": 1, "position": 80, "queries": 5586, "updates": 5, "loss": 0.3305647075176239, "gradient_norm": 2.8144664764404297}
{"stage": "train", "epoch": 1, "position": 96, "queries": 5586, "updates": 6, "loss": 0.3088882565498352, "gradient_norm": 3.2798023223876953}
{"stage": "train", "epoch": 1, "position": 112, "queries": 5586, "updates": 7, "loss": 0.34250226616859436, "gradient_norm": 2.5037524700164795}


{'checkpoint_sha256': {'epoch-1.pt': '2daa5f1bf50e6809b3998a4e5fa8ee958e83fdc8e0317fa8219350eb8a4dd8fc',
  'epoch-2.pt': 'ef4c293ba78917522c81fa119a7406c3c936330c0985b53e5a0188cf91e36cc6'},
 'contract_hash': '1a0a2db621d773d401527c540b9bc8d3648f7d34c3424f0049dda6b0f5fea42d',
 'duplicate_exclusions': ['114846',
  '117908',
  '139536',
  '156640',
  '61406',
  '63562',
  '77610'],
 'epochs': 2,
 'held_fold': 'fold_0',
 'peak_allocated_mib': 3592.21875,
 'peak_reserved_mib': 3648.0,
 'runtime_seconds_this_process': 570.409222512,
 'schema_version': 'dsc2026.research_v2.exp112_e5_transfer_checkpoint.v1',
 'scientific_contract': {'data': 'f7dbe26debfc71ed42aee171a77876967c6a5e992999085d9513c81e3d72e013',
  'epochs': 2,
  'mechanism': 'exact_exp112_query_only_v1',
  'microbatch': 4,
  'qids': ['96',
   '294',
   '458',
   '564',
   '626',
   '636',
   '728',
   '816',
   '818',
   '876',
   '974',
   '1146',
   '1186',
   '1264',
   '1274',
   '1674',
   '1870',
   '1876',
   '2162',
   '219

In [12]:
score_out = OUT / 'score'
checkpoint = train_out / 'epoch-2.pt'
subprocess.run([sys.executable, str(runner), 'score-fold0', '--bundle', str(BUNDLE), '--output', str(score_out), '--checkpoint', str(checkpoint)], check=True)
report = json.loads((score_out / 'E5_TRANSFER_FOLD0_PILOT_REPORT.json').read_text(encoding='utf-8'))
assert report['full_five_fold_launched'] is False
receipt = {
    'status': 'FOLD0_COMPLETE_STOP_FOR_USER_REVIEW',
    'manifest_sha256': sha256(BUNDLE / 'E5_TRANSFER_INPUT_MANIFEST.json'),
    'runner_sha256': sha256(runner),
    'checkpoint_sha256': sha256(checkpoint),
    'pilot_report_sha256': sha256(score_out / 'E5_TRANSFER_FOLD0_PILOT_REPORT.json'),
    'predictions_sha256': sha256(score_out / 'E5_TRANSFER_FOLD0_PREDICTIONS.jsonl'),
    'runtime': {'python': sys.version, 'platform': platform.platform(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'peft': peft.__version__, 'gpu': torch.cuda.get_device_name(0)},
    'full_five_fold_launcher_present': False,
}
(OUT / 'KAGGLE_RUN_RECEIPT.json').write_text(json.dumps(receipt, indent=2, sort_keys=True) + '\n', encoding='utf-8')
print(json.dumps(report, indent=2, ensure_ascii=False))
print('STOP: download output and return it for user review. No five-fold job is defined in this notebook.')


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 2615.02it/s]


{"stage": "score", "completed": 25, "total": 1398}
{"stage": "score", "completed": 50, "total": 1398}
{"stage": "score", "completed": 75, "total": 1398}
{"stage": "score", "completed": 100, "total": 1398}
{"stage": "score", "completed": 125, "total": 1398}
{"stage": "score", "completed": 150, "total": 1398}
{"stage": "score", "completed": 175, "total": 1398}
{"stage": "score", "completed": 200, "total": 1398}
{"stage": "score", "completed": 225, "total": 1398}
{"stage": "score", "completed": 250, "total": 1398}
{"stage": "score", "completed": 275, "total": 1398}
{"stage": "score", "completed": 300, "total": 1398}
{"stage": "score", "completed": 325, "total": 1398}
{"stage": "score", "completed": 350, "total": 1398}
{"stage": "score", "completed": 375, "total": 1398}
{"stage": "score", "completed": 400, "total": 1398}
{"stage": "score", "completed": 425, "total": 1398}
{"stage": "score", "completed": 450, "total": 1398}
{"stage": "score", "completed": 475, "total": 1398}
{"stage": "scor